# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading, exploring, and preparing the FAIR^2 dataset using the `mlcroissant` library and Croissant metadata schema. All references to record sets, fields, and data schema elements are given via their Croissant `@id` values for robustness and traceability.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We load the Croissant dataset using the provided schema URL. Metadata access provides information about available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata_dict = json.loads(dataset.metadata.to_jsonld())  # Full JSON-LD metadata as dict

# Print dataset name and description
print("Dataset:", dataset.metadata.name)
print("Description:", dataset.metadata.description)

## 2. Data Overview

We inspect available records by examining record set `@id`s, their fields, and associated columns. Croissant's design allows us to identify these entities unambiguously.

**Listing record sets, their `@id`s, fields, and example columns:**

In [ ]:
# Helper to find @id entries of all record sets, fields, and columns
def extract_record_sets(md):
    """Given full metadata JSON, returns list of recordSet dicts."""
    # Handles flattened or graph-structured JSON-LD
    def _objects():
        if isinstance(md, dict) and '@graph' in md:
            return md['@graph']
        elif isinstance(md, list):
            return md
        else:
            return [md]
    return [o for o in _objects() if o.get('@type') in ['cr:RecordSet', 'RecordSet']]

def extract_fields(md):
    def _objects():
        if isinstance(md, dict) and '@graph' in md:
            return md['@graph']
        elif isinstance(md, list):
            return md
        else:
            return [md]
    return [o for o in _objects() if o.get('@type') in ['cr:Field', 'Field']]

# Find all record sets
record_sets = extract_record_sets(metadata_dict)

print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    rec_id = rs.get('@id')
    print(f"- RecordSet @id: {rec_id}")
    fields = rs.get('cr:field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields by @id:")
    for f in fields:
        fid = f.get('@id') if isinstance(f, dict) else f
        print(f"     - {fid}")
    sources = rs.get('cr:source', [])
    if isinstance(sources, dict):
        sources = [sources]
    print("  Data sources by @id:")
    for s in sources:
        sid = s.get('@id') if isinstance(s, dict) else s
        print(f"     - {sid}")
    print("")

# For further use, collect @id of first record set
default_record_set_id = record_sets[0]['@id'] if record_sets else None

## 3. Data Extraction

We load all records from the (first/main) record set into a pandas DataFrame. The data structure is established via record set and field `@id`s only.

You can repeat this process for all listed record sets if more exist.

In [ ]:
if not record_sets:
    raise RuntimeError("No RecordSets found in the dataset's metadata.")

# For demonstration, load the first record set by its @id
rs_id = record_sets[0]['@id']
print(f"Loading records from RecordSet @id: {rs_id}")

# Retrieve all records as dictionaries and construct a DataFrame
records = list(dataset.records(record_set=rs_id))
df = pd.DataFrame(records)

print("DataFrame columns:", df.columns.tolist())
display(df.head())

## 4. Exploratory Data Analysis (EDA)

Apply data processing: filter on a numeric column by its `@id`, normalize values, and group data by a categorical column. 

You must reference all fields by their `@id` only (see Section 2 output for available ids). If unsure, use one likely to be numeric (e.g. molecular weight, peptide count, or abundance).


In [ ]:
# Example: select first numeric field for analysis by inspecting col names
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    # Try to heuristically pick possible numeric columns
    possible_numeric = [col for col in df.columns if any(s in col.lower() for s in ['mw', 'mass', 'weight', 'count', 'abund', 'peptide', 'ratio', 'molecular'])]
    if possible_numeric:
        numeric_field = possible_numeric[0]
    else:
        raise RuntimeError("No numeric field found for EDA!")
else:
    numeric_field = numeric_fields[0]

print(f"Numeric field selected: {numeric_field}")

# For demonstration, use a threshold value relevant for the chosen column
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} entries")
display(filtered_df.head())

# Normalize the numeric field
col_norm = f"{numeric_field}_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First 5 normalized records for {numeric_field}:")
display(filtered_df[[numeric_field, col_norm]].head())

# Group by a string/categorical field
cat_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
if cat_fields:
    group_field = cat_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped average {numeric_field} by {group_field} (top 5):")
    print(grouped_df.head())
else:
    print("No categorical field available for grouping.")

## 5. Visualization

We plot the distribution of the selected numeric field and, if available, relationships with a group field. All field references are strictly via column `@id` names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If categorical group field available, show group means
if cat_fields:
    plt.figure(figsize=(10,4))
    order = filtered_df[group_field].value_counts().index[:10]
    sns.barplot(y=group_field, x=numeric_field, data=filtered_df, order=order, ci=None)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.show()

## 6. Conclusion

We loaded and explored record-level data defined by FAIR^2 Croissant package [@id references only].

- Inspected record set and field structure programmatically using Croissant `@id`s only.
- Loaded all records into pandas, filtered and normalized a numeric field, and grouped by a categorical field.
- Visualized numeric distributions and group-level differences.

To further process or analyze the data, simply select additional record sets/fields by their `@id`s as shown above.